# RelPoseNet + Focus-point losses (Hard / Soft / SoftGrid) — Colab runner
هر سلول را به ترتیب اجرا کنید. همهٔ کد در پوشهٔ `relposenet/` همین zip است؛ این نوت‌بوک فقط مسیرها را می‌گذارد و اجرا می‌کند.
اجرا resumable است (checkpoint روی Drive): اگر Colab قطع شد، سلول «اجرای لاس‌ها» را دوباره بزنید.

In [ ]:
# ---- 1) مسیرها و آماده‌سازی ----
from google.colab import drive
drive.mount("/content/drive")

ZIP = "/content/drive/MyDrive/RelPoseNet_losses.zip"          # ← مسیر همین zip روی Drive
SRC_IMG_DIR = "/content/drive/MyDrive/7scenes-light"           # ← 7-Scenes ریسایزشده (ضلع کوچک ۲۵۶) شامل *.color.png و *.pose.txt
OUT_DIR = "/content/drive/MyDrive/relpose_runs"                # نتایج/چک‌پوینت‌ها
WORK = "/content/RelPoseNet_TEST-main"
IMG_DIR = "/content/7scenes-light"                             # کپی محلی (سریع‌تر از Drive)

!unzip -q -o "{ZIP}" -d /content
!pip -q install pandas matplotlib tqdm pillow scipy
import sys; sys.path.insert(0, WORK)

In [ ]:
# ---- 2) تست‌های درستی (برچسب‌ها و لاس‌ها؛ بدون داده) ----
!cd {WORK} && python tests/test_labels.py && python tests/smoke_test_losses.py

In [ ]:
# ---- 3) کپی داده به دیسک محلی + آماده‌سازی + بررسی قرارداد برچسب ----
import os, shutil, numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from PIL import Image
from relposenet.trainer import make_cfg, prepare_data
from relposenet.pose_dataset import read_pair_file, read_test_pairs, unique_paths
from relposenet.labels import check_label_convention

cfg = make_cfg(WORK, IMG_DIR, OUT_DIR)

if SRC_IMG_DIR != IMG_DIR:
    d = cfg.data
    recs = (read_pair_file(d.train_pairs, SRC_IMG_DIR) + read_pair_file(d.val_pairs, SRC_IMG_DIR)
            + read_test_pairs(d.test_pairs, SRC_IMG_DIR, n=cfg.output.test_n)[0])
    paths = unique_paths(recs)
    def cp(p):
        for suf in (".color.png", ".pose.txt"):
            s = p.replace(".color.png", suf); dst = IMG_DIR + s[len(SRC_IMG_DIR):]
            if os.path.exists(dst) or not os.path.exists(s): continue
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copyfile(s, dst)
    with ThreadPoolExecutor(32) as ex: list(tqdm(ex.map(cp, paths), total=len(paths)))

data = prepare_data(cfg)       # اگر تصویر یا فایل pose کم باشد، با پیام روشن متوقف می‌شود
# اگر pose ندارید: فقط MSE اصلی قابل اجراست -> cfg.data.use_poses = False و فقط "RelPose-MSE@world"

w, h = Image.open(data["train"][0]["p1"]).size
cfg.loss.focal_px = 525.0 * h / 480.0                      # Kinect RGB f=525px در 640x480 (تقریبی؛ کالیبره نیست)
print(f"image size (w,h) = {(w, h)} -> focal_px = {cfg.loss.focal_px:.1f}")
if min(w, h) != 256: print("⚠ تصاویر باید با ضلع کوچک ۲۵۶ ریسایز شده باشند (طبق README)")

et, eq = check_label_convention(data["train"], data["poses"])
print("translation hypotheses:", {k: round(v, 6) for k, v in sorted(et.items(), key=lambda x: x[1])[:3]})
print("rotation hypotheses   :", {k: round(v, 6) for k, v in sorted(eq.items(), key=lambda x: x[1])[:2]})
assert min(et, key=et.get) == "world(C2-C1)" and et["world(C2-C1)"] < 1e-4, "قرارداد t با فرض من نمی‌خواند"
assert min(eq, key=eq.get) == "R1^T R2" and eq["R1^T R2"] < 1e-4, "قرارداد q با فرض من نمی‌خواند"
print("✔ label convention confirmed: t = C2-C1 (world), q = R1^T R2")

In [ ]:
# ---- 4) سرعت و تعداد iteration ----
from relposenet.trainer import benchmark_speed
ips = benchmark_speed(cfg, data, "Hard(zf=1)@cam1")        # زمان واقعی یک run کامل را چاپ می‌کند

def set_iters(cfg, n):                                      # کاهش متناسب برنامهٔ lr و فاصلهٔ validation
    r = n / 125000
    cfg.train.n_train_iters = n
    cfg.train.lrate_decay_steps = max(1, round(15000 * r))
    cfg.output.validate_interval = max(200, round(1300 * r))

N_ITERS = 125000        # ← مقدار اصلی کانفیگ نویسنده. اگر زمان بالا بود مثلاً 20000 بگذارید
set_iters(cfg, N_ITERS)
# اگر RAM بالای ~۲۰ گیگ دارید: cfg.data.ram_cache = True ; data = prepare_data(cfg)  (دیکد PNG را حذف می‌کند)
# cfg.loss.symmetric = True  -> لاس‌های هندسی میانگین دو لنگر (دوربین ۱ و ۲ به‌عنوان مرجع) می‌شوند

In [ ]:
# ---- 5) اجرای لاس‌ها (هر کدام جدا، resumable) ----
# قالب نام: <loss>@<frame>   frame = cam1 (ترجمه در دستگاه دوربین ۱؛ لازم برای لاس‌های هندسی) یا world (برچسب اصلی نویسنده)
# هر عضو خانواده‌ها آزاد است: Hard(zf=..), Soft(zf=..,s=..), SoftGrid(zf=..,s=..,r=..), SoftSqrt(..), SoftGridSqrt(..),
# SE3-geo, Ray-Plucker, DenseRay, P-MSE, PoseNet(b=..), RelPose-MSE
from relposenet.trainer import run_training
SPECS = ["RelPose-MSE@world",             # لاس و قاب اصلی نویسنده
         "RelPose-MSE@cam1",              # همان MSE با برچسب دستگاه دوربین ۱
         "SE3-geo@cam1",
         "Hard(zf=1)@cam1",
         "Soft(zf=1,s=4)@cam1",
         "SoftGrid(zf=1,s=4,r=48)@cam1"]
SEEDS = [cfg.seed]                         # برای اطمینان آماری حداقل ۳ seed برای ۲-۳ لاس برتر
for seed in SEEDS:
    for spec in SPECS:
        run_training(cfg, spec, data, seed=seed)

In [ ]:
# ---- 6) نمودار آموزش، جدول دقت نهایی و مقایسهٔ جفتی ----
from relposenet.report import results_table, plot_training, paired_compare, author_reference
from relposenet.trainer import summarize
from relposenet.labels import rel_labels
import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 30)

ref = author_reference(cfg, data)           # مدل نویسنده روی همان جفت‌های تست
rot, tn = [], []
for r in data["test"]:
    (R1, C1), (R2, C2) = data["poses"][r["p1"]], data["poses"][r["p2"]]; l = rel_labels(R1, C1, R2, C2)
    rot.append(np.degrees(np.arccos(np.clip((np.trace(l["R"]) - 1) / 2, -1, 1)))); tn.append(np.linalg.norm(l["t_cam"]))
triv = summarize(dict(rot=np.array(rot), t=np.array(tn), tdir=np.full(len(rot), 90.0), loss=np.zeros(len(rot))))
print("trivial (identity rot, zero t): rot_med %.1f deg | t_med %.3f m" % (triv["rot_med"], triv["t_med"]))

print("\n=== TEST pairs, last weights (sorted by se3_mean = t + rot[rad]) ===")
print(results_table(OUT_DIR, "test", "last", reference=ref).round(3).to_string(index=False))
print("\n=== TEST pairs, best-val weights ===");  print(results_table(OUT_DIR, "test", "best").round(3).to_string(index=False))
print("\n=== VAL (نشتی: فریم‌ها در train هم هستند) ==="); print(results_table(OUT_DIR, "val", "last").round(3).to_string(index=False))

plot_training(OUT_DIR)
print(paired_compare(OUT_DIR, "RelPose-MSE@cam1", [s for s in SPECS if s != "RelPose-MSE@cam1"], seed=cfg.seed).round(4).to_string(index=False))